# Прогон A — `feature/eval-inference`

Secrets: `GIT_URL`, `GITHUB_TOKEN`, `GIT_BRANCH=feature/eval-inference`.

Accelerator: **GPU T4** 

Add Data: competition `filament-segmentation-2026`.
После train — Save Version **с Output**, `last.pt`.

In [1]:
import os, sys, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
for name in ("GIT_URL", "GITHUB_TOKEN", "GIT_BRANCH"):
    try:
        os.environ[name] = sec.get_secret(name)
    except Exception:
        pass

REPO = Path("/kaggle/working/filament-segmentation-2026")
BRANCH = os.environ.get("GIT_BRANCH", "feature/eval-inference")
url = os.environ.get("GIT_URL", "")
token = os.environ.get("GITHUB_TOKEN", "")
if token and url.startswith("https://"):
    url = url.replace("https://", f"https://{token}@", 1)

if not (REPO / "run.py").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)])
else:
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"])

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True))
!pip -q install transformers accelerate safetensors

Cloning into '/kaggle/working/filament-segmentation-2026'...


74f4a6d add inference evaluation and threshold sweep



In [2]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Settings → GPU T4, Restart session"
x = torch.zeros(1, device="cuda")
print("ok", x.device)

True Tesla T4
ok cuda:0


In [3]:
%cd /kaggle/working/filament-segmentation-2026
!python run.py check
!python run.py train --size 1024 --epochs 8
!ls -lh /kaggle/working/outputs/m2f/last.pt

/kaggle/working
check ok
preprocessor_config.json: 100%|████████████████| 537/537 [00:00<00:00, 2.39MB/s]
/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: '_max_size', 'reduce_labels'
  image_processor = cls(**image_processor_dict)
config.json: 79.8kB [00:00, 63.2MB/s]
model.safetensors: 100%|█████████████████████| 190M/190M [00:03<00:00, 58.6MB/s]
Loading weights: 100%|█| 566/566 [00:00<00:00, 1474.41it/s, Materializing param=
Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-coco-instance
Key                    | Status   |                                                                                      
-----------------------+----------+--------------------------------------------------------------------------------------
criterion.empty_weight | MISMATCH | Reinit due to size mismatch ckpt: torch

In [4]:
%cd /kaggle/working/filament-segmentation-2026
!python run.py sweep --weights /kaggle/working/outputs/m2f/last.pt

/kaggle/working
/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: '_max_size', 'reduce_labels'
  image_processor = cls(**image_processor_dict)
Loading weights: 100%|█| 566/566 [00:00<00:00, 1580.44it/s, Materializing param=
Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-coco-instance
Key                    | Status   |                                                                                      
-----------------------+----------+--------------------------------------------------------------------------------------
criterion.empty_weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([81]) vs model:torch.Size([2])          
class_predictor.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([81, 256]) vs model:torch.Size([2, 256])
class_predictor.bias   | MISMATCH | Rein

In [5]:
%cd /kaggle/working/filament-segmentation-2026
!python run.py infer --weights /kaggle/working/outputs/m2f/last.pt --score-thr 0.4 --min-area 80 --viz-worst 12 --submit

/kaggle/working
/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: '_max_size', 'reduce_labels'
  image_processor = cls(**image_processor_dict)
Loading weights: 100%|█| 566/566 [00:00<00:00, 1470.64it/s, Materializing param=
Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-coco-instance
Key                    | Status   |                                                                                      
-----------------------+----------+--------------------------------------------------------------------------------------
criterion.empty_weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([81]) vs model:torch.Size([2])          
class_predictor.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([81, 256]) vs model:torch.Size([2, 256])
class_predictor.bias   | MISMATCH | Rein